#### Correr la imagen y levantar contenedor en docker
- docker run -d --name mongo -p 27017:27017 mongo:8.0

#### Crear proyecto
- uv init --no-package intro_mongodb --python 3.12

#### Instalar dependencias
- uv add ipykernel pymongo

In [1]:
import pymongo

print(pymongo.__version__)

4.19.0


In [2]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")
client.admin.command("ping")

{'ok': 1.0}

In [3]:
# Creando DB
db = client["escuela"]

In [4]:
# Creando colección
alumnos = db["alumnos"]

### Create

In [5]:
# CRUD -> Create / Read / Update / Delete
# Insertando un documento a la colección
res = alumnos.insert_one(
    {
        "nombre": "Ana Torres",
        "edad": 21,
        "ciudad": "Lima",
        "cursos": ["Python", "SQL"]
    }
)

print(res.inserted_id)

6ae2c4db1cd381e45040a537


In [6]:
alumnos.insert_many(
    [
        {
            "nombre": "Luis Ramos",
            "edad": 19,
            "ciudad": "Bogotá"
        },
        {
            "nombre": "María Paz",
            "edad": 23,
            "ciudad": "Lima",
            "beca": True
        }
    ]
)

InsertManyResult([ObjectId('6ae2c4db1cd381e45040a538'), ObjectId('6ae2c4db1cd381e45040a539')], acknowledged=True)

### Read / LECTURAS

In [7]:
# CRUD -> Create / Read / Update / Delete

# EQUIVALENTE ---> alumnos.find() -> SELECT * FROM alumnos
for alumno in alumnos.find():
    print(alumno)
    print("-" * 50)

{'_id': ObjectId('6ae2a84b2adea2708e930bd6'), 'nombre': 'Ana Torres', 'edad': 18, 'ciudad': 'Arequipa', 'cursos': ['SQL', 'MongoDB']}
--------------------------------------------------
{'_id': ObjectId('6ae2a8c02adea2708e930bd7'), 'nombre': 'Luis Ramos', 'edad': 16, 'ciudad': 'Bogotá'}
--------------------------------------------------
{'_id': ObjectId('6ae2a8c02adea2708e930bd8'), 'nombre': 'María Paz', 'edad': 20, 'ciudad': 'Lima', 'beca': True}
--------------------------------------------------
{'_id': ObjectId('6ae2c4db1cd381e45040a537'), 'nombre': 'Ana Torres', 'edad': 21, 'ciudad': 'Lima', 'cursos': ['Python', 'SQL']}
--------------------------------------------------
{'_id': ObjectId('6ae2c4db1cd381e45040a538'), 'nombre': 'Luis Ramos', 'edad': 19, 'ciudad': 'Bogotá'}
--------------------------------------------------
{'_id': ObjectId('6ae2c4db1cd381e45040a539'), 'nombre': 'María Paz', 'edad': 23, 'ciudad': 'Lima', 'beca': True}
--------------------------------------------------


#### EQUIVALENTES

In [8]:
# EQUIVALENTE ---> alumnos.find({"ciudad": "Lima"}) -> SELECT * FROM alumnos WHERE ciudad = "Lima"
for alumno in alumnos.find({"ciudad": "Bogotá"}):
    print(alumno)
    print("-" * 50)

{'_id': ObjectId('6ae2a8c02adea2708e930bd7'), 'nombre': 'Luis Ramos', 'edad': 16, 'ciudad': 'Bogotá'}
--------------------------------------------------
{'_id': ObjectId('6ae2c4db1cd381e45040a538'), 'nombre': 'Luis Ramos', 'edad': 19, 'ciudad': 'Bogotá'}
--------------------------------------------------


In [9]:
# alumnos.find({"ciudad": "Lima"}) -> SELECT * FROM alumnos WHERE ciudad = "Lima" LIMIT 1
res = alumnos.find_one({"ciudad": "Lima"})
res
# for field in res:
#     print(f"key: {field} -> Valor: {res[field]}")

{'_id': ObjectId('6ae2a8c02adea2708e930bd8'),
 'nombre': 'María Paz',
 'edad': 20,
 'ciudad': 'Lima',
 'beca': True}

In [10]:
# alumnos.find({}, {"nombre": 1, "edad": 1}) -> SELECT nombre, edad FROM alumnos
# "_id": 0 ---> No muestra id
for alumno in alumnos.find({}, {"nombre": 1, "edad": 1, "_id": 0}):
    print(alumno)

{'nombre': 'Ana Torres', 'edad': 18}
{'nombre': 'Luis Ramos', 'edad': 16}
{'nombre': 'María Paz', 'edad': 20}
{'nombre': 'Ana Torres', 'edad': 21}
{'nombre': 'Luis Ramos', 'edad': 19}
{'nombre': 'María Paz', 'edad': 23}


In [11]:
# alumnos.find().sort("edad": -1) -> SELECT * FROM alumnos ORDER BY edad DESC
# -1 = DESC y 1 = ASCEN
for alumno in alumnos.find({}).sort({"edad": -1}):
    print(alumno)

{'_id': ObjectId('6ae2c4db1cd381e45040a539'), 'nombre': 'María Paz', 'edad': 23, 'ciudad': 'Lima', 'beca': True}
{'_id': ObjectId('6ae2c4db1cd381e45040a537'), 'nombre': 'Ana Torres', 'edad': 21, 'ciudad': 'Lima', 'cursos': ['Python', 'SQL']}
{'_id': ObjectId('6ae2a8c02adea2708e930bd8'), 'nombre': 'María Paz', 'edad': 20, 'ciudad': 'Lima', 'beca': True}
{'_id': ObjectId('6ae2c4db1cd381e45040a538'), 'nombre': 'Luis Ramos', 'edad': 19, 'ciudad': 'Bogotá'}
{'_id': ObjectId('6ae2a84b2adea2708e930bd6'), 'nombre': 'Ana Torres', 'edad': 18, 'ciudad': 'Arequipa', 'cursos': ['SQL', 'MongoDB']}
{'_id': ObjectId('6ae2a8c02adea2708e930bd7'), 'nombre': 'Luis Ramos', 'edad': 16, 'ciudad': 'Bogotá'}


In [12]:
# alumnos.count_documents({}) -> SELECT COUNT(1) FROM alumnos
print(alumnos.count_documents({}))

# alumnos.count_documents({"edad": 21}) -> SELECT COUNT(1) FROM alumnos WHERE edad = 21
print(alumnos.count_documents({"ciudad": "Lima"}))

6
3


In [13]:
res = alumnos.insert_one(
    {
        "nombre": "Marisol Vásquez",
        "edad": 30,
        "direccion": {
            "ciudad": "Lima",
            "distrito": "Miraflores"
        },
        "cursos": ["Python", "MongoDB"]
    }
)

In [14]:
for alumno in alumnos.find({"direccion.distrito": "Miraflores"}):
    print(alumno)

{'_id': ObjectId('6ae2c4db1cd381e45040a53a'), 'nombre': 'Marisol Vásquez', 'edad': 30, 'direccion': {'ciudad': 'Lima', 'distrito': 'Miraflores'}, 'cursos': ['Python', 'MongoDB']}


In [15]:
# igual, no igual
# MongoDB -> $eq $ne
# SQL -> = / <> ó !=

for alumno in alumnos.find({
    "ciudad": {"$ne": "Lima"},
    "direccion.ciudad": {"$ne": "Lima"}
}):
    print(alumno)

{'_id': ObjectId('6ae2a84b2adea2708e930bd6'), 'nombre': 'Ana Torres', 'edad': 18, 'ciudad': 'Arequipa', 'cursos': ['SQL', 'MongoDB']}
{'_id': ObjectId('6ae2a8c02adea2708e930bd7'), 'nombre': 'Luis Ramos', 'edad': 16, 'ciudad': 'Bogotá'}
{'_id': ObjectId('6ae2c4db1cd381e45040a538'), 'nombre': 'Luis Ramos', 'edad': 19, 'ciudad': 'Bogotá'}


In [16]:
# igual, no igual
# MongoDB -> $eq $ne
# SQL -> = / <> ó !=

# gt -> greater than (>)
# gte -> greater than equal (>=)
# lt -> less than (<)
# lte -> less than equal (<=)

for alumno in alumnos.find({
    "edad": {"$gt": 19}
}):
    print(alumno)

{'_id': ObjectId('6ae2a8c02adea2708e930bd8'), 'nombre': 'María Paz', 'edad': 20, 'ciudad': 'Lima', 'beca': True}
{'_id': ObjectId('6ae2c4db1cd381e45040a537'), 'nombre': 'Ana Torres', 'edad': 21, 'ciudad': 'Lima', 'cursos': ['Python', 'SQL']}
{'_id': ObjectId('6ae2c4db1cd381e45040a539'), 'nombre': 'María Paz', 'edad': 23, 'ciudad': 'Lima', 'beca': True}
{'_id': ObjectId('6ae2c4db1cd381e45040a53a'), 'nombre': 'Marisol Vásquez', 'edad': 30, 'direccion': {'ciudad': 'Lima', 'distrito': 'Miraflores'}, 'cursos': ['Python', 'MongoDB']}


### Update

In [17]:
alumnos.update_one(
    {"nombre": "Ana Torres"}, # filtro
    {"$set": {"ciudad": "Arequipa"}} # fija o establece
)

alumnos.find_one({"nombre": "Ana Torres"})

{'_id': ObjectId('6ae2a84b2adea2708e930bd6'),
 'nombre': 'Ana Torres',
 'edad': 18,
 'ciudad': 'Arequipa',
 'cursos': ['SQL', 'MongoDB']}

In [18]:
alumnos.update_many({}, {"$inc": {"edad": -1}})
alumnos.find_one({"nombre": "Ana Torres"})

{'_id': ObjectId('6ae2a84b2adea2708e930bd6'),
 'nombre': 'Ana Torres',
 'edad': 17,
 'ciudad': 'Arequipa',
 'cursos': ['SQL', 'MongoDB']}

In [19]:
alumnos.find_one({"nombre": "Marisol Vásquez"})

{'_id': ObjectId('6ae2c4db1cd381e45040a53a'),
 'nombre': 'Marisol Vásquez',
 'edad': 29,
 'direccion': {'ciudad': 'Lima', 'distrito': 'Miraflores'},
 'cursos': ['Python', 'MongoDB']}

In [20]:
alumnos.update_one(
    {"nombre": "Ana Torres"}, # filtro
    {"$push": {"cursos": "MongoDB"}} # adicionar elementos a un arreglo
)

alumnos.find_one({"nombre": "Ana Torres"})

{'_id': ObjectId('6ae2a84b2adea2708e930bd6'),
 'nombre': 'Ana Torres',
 'edad': 17,
 'ciudad': 'Arequipa',
 'cursos': ['SQL', 'MongoDB', 'MongoDB']}

In [21]:
alumnos.update_one(
    {"nombre": "Ana Torres"}, # filtro
    {"$pull": {"cursos": "Python"}} # eliminar elementos a un arreglo
)

alumnos.find_one({"nombre": "Ana Torres"})

{'_id': ObjectId('6ae2a84b2adea2708e930bd6'),
 'nombre': 'Ana Torres',
 'edad': 17,
 'ciudad': 'Arequipa',
 'cursos': ['SQL', 'MongoDB', 'MongoDB']}

### Delete

In [22]:
list(alumnos.find())

[{'_id': ObjectId('6ae2a84b2adea2708e930bd6'),
  'nombre': 'Ana Torres',
  'edad': 17,
  'ciudad': 'Arequipa',
  'cursos': ['SQL', 'MongoDB', 'MongoDB']},
 {'_id': ObjectId('6ae2a8c02adea2708e930bd7'),
  'nombre': 'Luis Ramos',
  'edad': 15,
  'ciudad': 'Bogotá'},
 {'_id': ObjectId('6ae2a8c02adea2708e930bd8'),
  'nombre': 'María Paz',
  'edad': 19,
  'ciudad': 'Lima',
  'beca': True},
 {'_id': ObjectId('6ae2c4db1cd381e45040a537'),
  'nombre': 'Ana Torres',
  'edad': 20,
  'ciudad': 'Lima',
  'cursos': ['Python', 'SQL']},
 {'_id': ObjectId('6ae2c4db1cd381e45040a538'),
  'nombre': 'Luis Ramos',
  'edad': 18,
  'ciudad': 'Bogotá'},
 {'_id': ObjectId('6ae2c4db1cd381e45040a539'),
  'nombre': 'María Paz',
  'edad': 22,
  'ciudad': 'Lima',
  'beca': True},
 {'_id': ObjectId('6ae2c4db1cd381e45040a53a'),
  'nombre': 'Marisol Vásquez',
  'edad': 29,
  'direccion': {'ciudad': 'Lima', 'distrito': 'Miraflores'},
  'cursos': ['Python', 'MongoDB']}]

In [23]:
from pymongo.collection import ObjectId

alumnos.delete_one({"_id": ObjectId("6ae2ab122adea2708e930bd9")})
list(alumnos.find())

[{'_id': ObjectId('6ae2a84b2adea2708e930bd6'),
  'nombre': 'Ana Torres',
  'edad': 17,
  'ciudad': 'Arequipa',
  'cursos': ['SQL', 'MongoDB', 'MongoDB']},
 {'_id': ObjectId('6ae2a8c02adea2708e930bd7'),
  'nombre': 'Luis Ramos',
  'edad': 15,
  'ciudad': 'Bogotá'},
 {'_id': ObjectId('6ae2a8c02adea2708e930bd8'),
  'nombre': 'María Paz',
  'edad': 19,
  'ciudad': 'Lima',
  'beca': True},
 {'_id': ObjectId('6ae2c4db1cd381e45040a537'),
  'nombre': 'Ana Torres',
  'edad': 20,
  'ciudad': 'Lima',
  'cursos': ['Python', 'SQL']},
 {'_id': ObjectId('6ae2c4db1cd381e45040a538'),
  'nombre': 'Luis Ramos',
  'edad': 18,
  'ciudad': 'Bogotá'},
 {'_id': ObjectId('6ae2c4db1cd381e45040a539'),
  'nombre': 'María Paz',
  'edad': 22,
  'ciudad': 'Lima',
  'beca': True},
 {'_id': ObjectId('6ae2c4db1cd381e45040a53a'),
  'nombre': 'Marisol Vásquez',
  'edad': 29,
  'direccion': {'ciudad': 'Lima', 'distrito': 'Miraflores'},
  'cursos': ['Python', 'MongoDB']}]